# 01 Factibilidad

Que fraccion de los valores del model-index aparece en el cuerpo del README.

In [1]:
import re
import pandas as pd
from collections import Counter
from pathlib import Path
from dotenv import load_dotenv

from metric_extraction.data import ref_values
from metric_extraction.download import safe_name
from metric_extraction.evidence import read_body, extract_hits, signature, table_rows
from metric_extraction.gold import gold_tuples, score_hits, value_close

load_dotenv()
READMES = Path("../data/raw/readmes")
df = pd.read_csv("../data/HFCO2_2026.csv", sep=",")

ref = df[df["performance_metrics"].notna()].copy()
ref["vals"] = ref["performance_metrics"].map(ref_values)
ref = ref[ref["vals"].notna()]

print(f"{len(df)} modelos | {len(list(READMES.glob('*.md')))} README en disco | {len(ref)} con referencia")

2057 modelos | 1931 README en disco | 393 con referencia


In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
HF_TOKEN = os.environ["HF_TOKEN"]

In [9]:
import pandas as pd
from pathlib import Path

from metric_extraction.download import safe_name
from metric_extraction.evidence import read_body, extract_hits

READMES = Path("../data/raw/readmes")
df = pd.read_csv("../data/HFCO2_2026.csv", sep=",")

# Modelos con performance_metrics vacío
EMPTY = {"", "[]", "{}", "null", "none", "nan"}
def is_empty(pm):
    return pd.isna(pm) or str(pm).strip().lower() in EMPTY

empty = df[df["performance_metrics"].map(is_empty)]
print(f"{len(empty)} de {len(df)} con performance_metrics vacío")
sample = empty.sample(min(20, len(empty)), random_state=42)

# Evidencia de métricas leyendo el README desde disco
rows = []
for _, r in sample.iterrows():
    p = READMES / f"{safe_name(r['modelId'])}.md"
    if not p.exists():                      # sin README: no cuenta como 0
        rows.append((r["modelId"], None, None, ""))
        continue
    text = read_body(p)
    hits = [h for h in extract_hits(text) if h[0] != "loss"]
    rows.append((r["modelId"], len(hits), len(text),
                 "; ".join(f"{k}={v}" for k, v in hits[:3])))

res = pd.DataFrame(rows, columns=["modelId", "n_hits", "largo_texto", "muestra"])
pd.set_option("display.width", 200, "display.max_colwidth", 70)
print(res.sort_values("n_hits"))

ok = res["n_hits"].notna()
print(f"\nREADME no encontrado: {(~ok).sum()} de {len(res)}")
print(f"Con evidencia de métricas: {(res['n_hits'].fillna(0) > 0).sum()} de {ok.sum()}")
print(f"Control: {res.loc[ok, 'muestra'].nunique()} muestras distintas y "
      f"{res.loc[ok, 'largo_texto'].nunique()} largos de texto distintos entre {ok.sum()} fichas")

1533 de 2057 con performance_metrics vacío
                                                          modelId  n_hits  largo_texto                                         muestra
4                                  DarwinAnim8or/GPT-NoSleep-355m     0.0       1284.0                                                
17                  BrianDsouzaAI/autotrain-tab-multi-92337144714     0.0        689.0                                                
13                       sindri101/radiology-report-en-zh-ft-base     1.0        476.0                                 sacrebleu=23.06
8                       Manolo/autotrain-nl-en-esco-1-78131140807     1.0        195.0                                sacrebleu=28.221
16                              Tritkoman/EnglishtoAncientGreekV4     1.0        194.0                                 sacrebleu=7.006
12                          wangdy/autotrain-godaddy2-38507101578     3.0        587.0                  r2=0.997; mse=0.042; mae=0.081
7   teacooki

In [2]:
import os, pandas as pd
from dotenv import load_dotenv
from metric_extraction.download import download_readmes

load_dotenv()
df = pd.read_csv("../data/HFCO2_2026.csv", sep=",")
status = download_readmes(df["modelId"].tolist(), "../data/raw/readmes",
                          token=os.environ["HF_TOKEN"])
print(status["result"].value_counts())
print(status["http"].value_counts())

100/1931
200/1931
300/1931
400/1931
500/1931
600/1931
700/1931
800/1931
900/1931
1000/1931
1100/1931
1200/1931
1300/1931
1400/1931
1500/1931
1600/1931
1700/1931
1800/1931
1900/1931
1931/1931
result
cached    1931
fail       126
Name: count, dtype: int64
http
200    1931
403      94
404      32
Name: count, dtype: int64


In [3]:
from pathlib import Path
from metric_extraction.evidence import read_body, signature, table_rows
from metric_extraction.download import safe_name

rows = []
for _, r in df.iterrows():
    p = Path("../data/raw/readmes") / f"{safe_name(r['modelId'])}.md"
    if not p.exists():
        continue
    b = read_body(p)
    rows.append((r["modelId"], r["pipeline_tag"], signature(b), table_rows(b)))

sig = pd.DataFrame(rows, columns=["modelId", "pipeline_tag", "firma", "filas_tabla"])
sig["tiene_firma"] = sig["firma"].map(len) > 0
print(f"{sig['tiene_firma'].mean():.1%} de {len(sig)} fichas con alguna métrica")

top = sig.loc[sig["tiene_firma"], "firma"].value_counts()
print(f"{len(top)} firmas distintas")
print((top.cumsum() / top.sum()).head(10).round(2))   # cobertura acumulada
print(sig.groupby("pipeline_tag")["tiene_firma"].agg(["mean", "size"])
         .sort_values("size", ascending=False).head(12))

87.1% de 1931 fichas con alguna métrica
35 firmas distintas
firma
(accuracy, macro f1, macro precision, macro recall, micro f1, micro precision, micro recall, weighted f1, weighted precision, weighted recall)                                                                                                                                        0.28
(accuracy, auc, f1, precision, recall)                                                                                                                                                                                                                                                0.47
(rouge1, rouge2, rougel, rougelsum)                                                                                                                                                                                                                                                   0.62
(accuracy, f1, precision, recall)                                                    

In [4]:
# 1) ¿De qué familias es la referencia?
ref = df[df["performance_metrics"].notna()
         & (df["performance_metrics"].astype(str).str.strip().str.len() > 2)]
print(ref["pipeline_tag"].value_counts().head(8))
print(ref["auto"].value_counts())

# 2) ¿Qué hay en los README de las tareas con 0 %?
import re
zero = sig[sig["pipeline_tag"].isin(["sentence-similarity", "feature-extraction",
                                     "translation", "text-generation"])
           & ~sig["tiene_firma"]].groupby("pipeline_tag").head(2)
for _, r in zero.iterrows():
    b = read_body(Path("../data/raw/readmes") / f"{safe_name(r['modelId'])}.md")
    lines = [l for l in b.splitlines() if re.search(r"\d\.\d{2,}", l)][:6]
    print("=====", r["pipeline_tag"], r["modelId"])
    print("\n".join(l[:140] for l in lines))

pipeline_tag
sentence-similarity     134
text-generation          94
feature-extraction       93
text-classification      77
text-ranking             33
image-classification     32
token-classification     14
summarization            13
Name: count, dtype: int64
auto
False    481
True      43
Name: count, dtype: int64
===== feature-extraction 1aurent/vit_base_patch16_224.owkin_pancancer
  - [Scaling Self-Supervised Learning for Histopathology with Masked Image Modeling](https://www.medrxiv.org/content/10.1101/2023.07.21.2329
@article{Filiot2023.07.21.23292757,
  elocation-id = {2023.07.21.23292757},
  doi          = {10.1101/2023.07.21.23292757},
  url          = {https://www.medrxiv.org/content/early/2023/09/14/2023.07.21.23292757},
  eprint       = {https://www.medrxiv.org/content/early/2023/09/14/2023.07.21.23292757.full.pdf},
===== text-generation Cran-May/mg

===== text-generation CreatedNull/MiniGPT

===== feature-extraction GrishaKushnir/jobbert-onnx
The model was presented in t

In [5]:
from pathlib import Path
from metric_extraction.evidence import read_body
from metric_extraction.download import safe_name
from metric_extraction.data import ref_values, value_in_text

ref = df[df["performance_metrics"].notna()].copy()
ref["vals"] = ref["performance_metrics"].map(ref_values)
ref = ref[ref["vals"].notna()]
print(len(ref), "fichas con valores de referencia")

out = []
for _, r in ref.iterrows():
    p = Path("../data/raw/readmes") / f"{safe_name(r['modelId'])}.md"
    if not p.exists():
        continue
    b = read_body(p)
    found = sum(value_in_text(v, b) for v in r["vals"])
    out.append((r["modelId"], r["pipeline_tag"], r["auto"], len(r["vals"]), found / len(r["vals"])))

cov = pd.DataFrame(out, columns=["modelId", "pipeline_tag", "auto", "n_vals", "frac"])
print(cov["frac"].describe().round(2))
print(f"{(cov['frac'] >= 0.5).mean():.1%} de las fichas con >=50% de sus valores en el cuerpo")
print(cov.groupby("pipeline_tag")["frac"].agg(["median", "size"])
         .sort_values("size", ascending=False).head(8).round(2))

393 fichas con valores de referencia
count    390.00
mean       0.86
std        0.29
min        0.00
25%        0.90
50%        1.00
75%        1.00
max        1.00
Name: frac, dtype: float64
89.7% de las fichas con >=50% de sus valores en el cuerpo
                      median  size
pipeline_tag                      
sentence-similarity     1.00   134
feature-extraction      0.92    92
text-generation         0.56    57
text-ranking            1.00    27
image-classification    1.00    27
text-classification     1.00    27
summarization           0.00    11
token-classification    1.00    10


In [6]:
import random
rng = random.Random(42)
ids = cov["modelId"].tolist()
null = []
for _, r in ref[ref["modelId"].isin(ids)].iterrows():
    other = rng.choice([i for i in ids if i != r["modelId"]])
    b = read_body(Path("../data/raw/readmes") / f"{safe_name(other)}.md")
    null.append(sum(value_in_text(v, b) for v in r["vals"]) / len(r["vals"]))
print(pd.Series(null).describe().round(2))

count    390.00
mean       0.09
std        0.18
min        0.00
25%        0.00
50%        0.00
75%        0.10
max        1.00
dtype: float64


In [7]:
from collections import Counter
from metric_extraction.evidence import read_body, inline_hits
from metric_extraction.gold import gold_tuples, score_hits, value_close

rows, miss = [], Counter()
for _, r in ref.iterrows():
    p = Path("../data/raw/readmes") / f"{safe_name(r['modelId'])}.md"
    if not p.exists():
        continue
    gold = gold_tuples(r["performance_metrics"])
    if not gold:
        continue
    hits = inline_hits(read_body(p))
    rec, n_gold, ok, n_hits = score_hits(hits, gold)
    rows.append((r["modelId"], r["pipeline_tag"], rec, n_gold, ok, n_hits))
    for al, rv in gold:                       # métricas de referencia no recuperadas
        if not any(n in al and value_close(v, rv) for n, v in hits):
            miss[sorted(al)[0]] += 1

sc = pd.DataFrame(rows, columns=["modelId", "pipeline_tag", "rec", "n_gold", "ok", "n_hits"])
tot = sc[["rec", "n_gold", "ok", "n_hits"]].sum()
P, R = tot["ok"] / tot["n_hits"], tot["rec"] / tot["n_gold"]
print(f"{len(sc)} fichas | P={P:.3f}  R={R:.3f}  F1={2*P*R/(P+R):.3f}")

g = sc.groupby("pipeline_tag").agg(cards=("modelId", "size"), rec=("rec", "sum"),
        n_gold=("n_gold", "sum"), ok=("ok", "sum"), n_hits=("n_hits", "sum"))
g["P"] = g["ok"] / g["n_hits"]; g["R"] = g["rec"] / g["n_gold"]
g["F1"] = 2 * g["P"] * g["R"] / (g["P"] + g["R"])
print(g[["cards", "P", "R", "F1"]].sort_values("cards", ascending=False).head(8).round(2))
print(miss.most_common(20))

390 fichas | P=0.977  R=0.371  F1=0.538
                      cards     P     R    F1
pipeline_tag                                 
sentence-similarity     134  0.98  0.33  0.49
feature-extraction       92  1.00  0.43  0.60
text-generation          57  0.18  0.01  0.01
text-ranking             27  1.00  0.36  0.52
image-classification     27  0.42  0.40  0.41
text-classification      27  0.68  0.32  0.44
summarization            11  0.02  0.02  0.02
token-classification     10   NaN  0.00   NaN
[('dot recall@3', 430), ('dot map@100', 430), ('dot recall@1', 420), ('dot recall@5', 418), ('dot ndcg@10', 414), ('dot mrr@10', 412), ('dot recall@10', 411), ('dot precision@5', 401), ('dot precision@3', 397), ('dot precision@10', 386), ('dot accuracy@3', 349), ('dot accuracy@1', 345), ('dot precision@1', 345), ('dot accuracy@5', 339), ('cosine map@100', 311), ('cosine recall@1', 304), ('cosine ndcg@10', 304), ('cosine recall@3', 303), ('dot accuracy@10', 301), ('cosine recall@5', 301)]
